Soup Kernel Challenge

Выберите GPU T4 x2 и включите интернет. Запустите все ячейки.


In [ ]:
SETUP_OK = False
import os, subprocess, sys, socket
for host in ('pypi.org', 'huggingface.co'):
    try:
        socket.getaddrinfo(host, 443)
    except OSError as exc:
        raise RuntimeError('Нет доступа к ' + host + '. Проверь Internet в настройках Kaggle.') from exc
gpus = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                      capture_output=True, text=True, check=True).stdout
print(gpus)
assert 'T4' in gpus, 'Выбери GPU T4 x2'
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass                                           
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0',
    'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'pytest'], check=True)
SETUP_OK = True
print('Окружение готово.')


In [ ]:
assert globals().get('SETUP_OK', False), 'Сначала должна успешно выполниться установка выше.'
import base64, gzip, hashlib, json, subprocess, sys, tempfile, zipfile
from pathlib import Path
from IPython.display import FileLink, display

                                                                              
PAYLOAD = '...'
EXPECTED = {'harness/_train.py': '45a62642dceab32cf3efcda504de87a9c25ea28b119b1806f56d14ff1146a89d', 'harness/check_rules.py': '879ab1861674f6e714eadd00aaf044235d98526dd75fab1035deec3401121ef5', 'harness/common.py': 'c83066f24f266dab62ebcc9c2d44f5a65902760b53da912e65ebfe511579c2db', 'harness/eval_adapter.py': '47cddbb8e4579c69f9535b099ff2371b1c3af08bfbd93d14bf79aadd74d38482', 'harness/example_reference.py': '6a11055f4a1822e53971c228c6e3e3aa6b88edd9eb6db81a388e26a70a030ceb', 'harness/run_task.py': 'd935a8959f348977313f2eef36870ff2699b9ba013a7978918ff35e148968a79', 'harness/score.py': '731c9b644dcc4069b0b069616f438064d1f7f376a202feb15201fd910ec688c5', 'LICENSE': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'NOTICE': '2f0089a88e9623e5f6eba497796f279c2fdb3a15be1df0973e4d0055d35876e4', 'README.md': 'ca1f681f8aeccc7fca9576a53a9798df668d672b3a2645160e4942d309e49a17', 'task/prepare_data.py': 'a8543f11a55aff80dbdc714a0e62391244a5296616eea2e34dc2eb3ff3ec18f7', 'task/task.yaml': 'dd70c739fcd143e06aef3bb1efcc9386f682bf1de26f30e08409d86a97ff47ce', 'tests/conftest.py': 'a82eabab468ea0ef1e003cccd7c3827ce2f4d00d09c32948a34fb48221f7337a', 'tests/test_callback.py': '1919303a3670c0a33abbfce9b33eab0913bb64a3a21c3312d86531c32b5bf490', 'tests/test_config.py': 'db808c44470d144b7cf18b7329492f41f76e37cfc24306e7cba0488bca9d787c', 'tests/test_formats.py': '5d656ed50c59f794e934473d459ea85635f892ba028e192414474e9b5d65919e', 'tests/test_loader.py': '3dad1bea2d33aab97e52b7cd44888d768bc9519af44fbe6cf3e96c260026632f', 'tests/test_trainer_init.py': 'b7608d62a40983bf7c9e5b270e0e527b641a837e462d6df2909fec88f607e821', 'solution/swiglu.py': 'e665910faeca6091c958ed7a191604484a44216f51d6fa069f5f48220e7472e6', 'solution/rmsnorm.py': '4329dd7f6af925f44a56392cddaff2b90a3d9cb03e122b209143d47961ba97d6', 'solution/entropy.py': '827a2d4ce8e08f0e7fd9fc80332a25154454f526e8dcfe36678f66df435eab9f', 'solution/teacher_nf4.py': 'f336b4b2271888159dc9e3b10831a023cfdd5f642b67456c3dc663ab7ad8d574', 'solution/teacher_grad_x.py': '82e40827ed18352493b8c05cc230b926e79b71e43045878781b7735160dd22c2', 'solution/__init__.py': 'b1bc238b9dcd8eeab6b09129d1a5cf565fec663c48f2a0a94eab7a14ce2d345d', 'solution/teacher_bridge.py': '4fa549c1730947f655d698728923ee5cc535baddbb61f5361c0e0f96a5722d8e'}
sources = json.loads(gzip.decompress(base64.b64decode(PAYLOAD)))
assert set(sources) == set(EXPECTED)
work = Path(tempfile.mkdtemp(prefix='soup-nf4-ce-', dir='/kaggle/working'))
for name, source in sources.items():
    assert hashlib.sha256(source.encode()).hexdigest() == EXPECTED[name], name
    target = (work / name).resolve()
    assert target.is_relative_to(work.resolve())
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
(work / 'source-manifest.json').write_text(json.dumps(EXPECTED, indent=2))
print('Исходники распакованы в', work)

def execute(args, logname, required=True):
    print('\nЗапуск:', ' '.join(args), flush=True)
    with (work / logname).open('w', encoding='utf-8') as log:
        with subprocess.Popen([sys.executable, '-u', *args], cwd=work,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = proc.wait()
    if required and code:
        raise RuntimeError(f'Остановлено: {logname}, код {code}. Пришли этот лог; обучение дальше не запускается.')
    return code

try:
                                                                                   
    print('Запуск проверки.')
    execute(['task/prepare_data.py'], 'prepare.log')
    execute(['harness/run_task.py'], 'benchmark.log')
    rules = execute(['harness/check_rules.py'], 'rules.log', required=False)
    score = execute(['harness/score.py', 'runs/result.json'], 'score.log', required=False)
    print('\nИТОГ:', 'Все проверки пройдены.' if rules == 0 and score == 0 else
          'Есть непройденные проверки. Это результат для разбора, не готовая сдача.')
finally:
    archive = Path('/kaggle/working') / (work.name + '-report.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in work.rglob('*'):
            if not p.is_file():
                continue
            rel = p.relative_to(work)
            if rel.parts[0] == 'data' or '__pycache__' in rel.parts:
                continue
            if p.suffix in ('.log', '.json', '.jsonl', '.yaml', '.yml', '.py'):
                z.write(p, rel.as_posix())
    print('\nАрхив с исходниками и результатами:', archive)
    display(FileLink(str(archive)))
